# 🎯 Module 12: Autoencoders

> **Stage:** Deep Learning  
> **Level:** Intermediate → Job-Ready  
> **Module:** Autoencoders  
> **Framework:** PyTorch  
> **Date:** 12 June 2026  
> **Previous:** 11 — Transfer Learning  
> **Next:** 13 — GANs

---

So far we've used neural networks to **predict labels**. Autoencoders use them to **learn compressed representations** of data — no labels required.

In this notebook we'll cover:

- What an autoencoder is
- Encoder, decoder, and the **latent space**
- **Bottleneck** / undercomplete architectures
- Reconstruction loss (MSE)
- Building a full autoencoder in PyTorch
- **Denoising autoencoders**
- **Anomaly detection** with reconstruction error
- Autoencoder vs **PCA**
- **Convolutional autoencoders**
- Overcomplete autoencoders and their pitfalls
- A brief connection to **VAEs**

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain the encoder → latent → decoder structure
- Describe what a **bottleneck** enforces and why it matters
- Compute MSE reconstruction loss
- Build and train a fully-connected autoencoder in PyTorch
- Extract and use the **latent representation**
- Build a **denoising autoencoder**
- Use reconstruction error for **anomaly detection**
- Compare autoencoders with **PCA**
- Build a **convolutional autoencoder** for images
- Explain why **overcomplete** autoencoders can be useless
- Describe at a high level what a **VAE** adds

---
## ⚙️ 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import time

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 🧠 1. What Is an Autoencoder?

An **autoencoder** is a neural network that learns to compress an input into a small representation and then reconstruct the original input from that representation.

### Basic structure

```text
Input
  ↓
Encoder
  ↓
Latent Representation
  ↓
Decoder
  ↓
Reconstructed Input
```

### Mathematically

$$x \;\rightarrow\; z \;\rightarrow\; \hat{x}$$

where

| Symbol | Meaning |
| --- | --- |
| $x$ | original input |
| $z$ | latent representation |
| $\hat{x}$ | reconstructed input |

### Goal

$$\hat{x} \approx x$$

### Key insight

The training target is the **input itself**. This is **self-supervised** learning — no labels required.

---
## 🎯 2. Why Use Autoencoders?

Useful for:

- **Dimensionality reduction**
- **Feature learning**
- **Data compression**
- **Denoising**
- **Anomaly detection**
- **Representation learning**
- **Pretraining**

### Example: compressing digits

```text
784 pixel values
      ↓
   Encoder
      ↓
32-dimensional latent vector
      ↓
   Decoder
      ↓
784 reconstructed pixels
```

Instead of hand-designing features, the network **learns** them from data.

---
## 🔧 3. Encoder, Latent Space, Decoder

### Encoder

$$z = f_\theta(x)$$

```text
Input:  784
   ↓
Linear: 256
   ↓
Linear: 64
   ↓
Latent: 16
```

### Latent Space

The internal representation learned by the encoder:

```text
784 → 128 → 32 → 8     ← final 8-dim = latent
```

A good latent space captures **important structure** and discards noise.

### Decoder

$$\hat{x} = g_\phi(z)$$

### Complete architecture

```text
784
 ↓
256
 ↓
64
 ↓
16   ← latent space (bottleneck)
 ↓
64
 ↓
256
 ↓
784
```

---
## 🍾 4. The Bottleneck — Why It Matters

The latent dimension is typically **smaller** than the input.

```text
784
 ↓
128
 ↓
32
 ↓
8    ← bottleneck
 ↓
32
 ↓
128
 ↓
784
```

### Why this is powerful

The network **cannot** simply copy the input through a smaller dimension.
It must learn a **compact, informative representation**.

This is called an **undercomplete autoencoder**.

⚠️ Without a bottleneck, the model can learn the identity mapping $x \to x$ and be useless for representation learning.

---
## 📉 5. Reconstruction Loss

A common choice is **Mean Squared Error**:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (x_i - \hat{x}_i)^2$$

Training minimizes:

$$\min_\theta \; L(x, \hat{x})$$

### Small numerical example

```text
Original:       [1.0, 0.5, 0.2]
Reconstruction: [0.8, 0.6, 0.3]
```

Squared errors:

$$(1.0 - 0.8)^2 = 0.04$$

$$(0.5 - 0.6)^2 = 0.01$$

$$(0.2 - 0.3)^2 = 0.01$$

$$\text{MSE} = \frac{0.04 + 0.01 + 0.01}{3} = 0.02$$

In [ ]:
# Verify the MSE example
x      = torch.tensor([1.0, 0.5, 0.2])
x_hat  = torch.tensor([0.8, 0.6, 0.3])
mse = F.mse_loss(x_hat, x)
print(f'MSE = {mse.item():.4f}')

# Equivalent manual calculation
manual = ((x - x_hat) ** 2).mean()
print(f'Manual = {manual.item():.4f}')

---
## 🐍 6. Building an Autoencoder in PyTorch

```text
784 → 128 → 32 → 16   (encoder)
16  → 32 → 128 → 784  (decoder)
```

We use a **sigmoid** output activation because MNIST pixels are normalized to `[0, 1]`.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, 32),  nn.ReLU(),
            nn.Linear(32, latent_dim),
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),  nn.ReLU(),
            nn.Linear(32, 128),         nn.ReLU(),
            nn.Linear(128, 784),        nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


# Shape check
ae = Autoencoder(latent_dim=16)
x = torch.randn(4, 784)
x_hat = ae(x)
print('Input :', x.shape)
print('Output:', x_hat.shape)
print('Latent:', ae.encoder(x).shape)
print('Params:', sum(p.numel() for p in ae.parameters()))

---
## 📦 7. Loading MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),   # flatten 28x28 → 784
])

train_ds = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)

print('Train:', len(train_ds), ' Test:', len(test_ds))
print('Sample shape:', train_ds[0][0].shape)

In [ ]:
# Show a few sample images
imgs, _ = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(16, 2.2))
for ax, img in zip(axes, imgs[:8]):
    ax.imshow(img.view(28, 28), cmap='gray')
    ax.axis('off')
plt.suptitle('MNIST samples')
plt.tight_layout(); plt.show()

---
## 🏋️ 8. Training the Autoencoder

Notice the crucial line: **the target is the input itself**.

```python
loss = criterion(reconstruction, x)
```

That is the essence of self-supervised learning with autoencoders.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, n = 0.0, 0
    for xb, _ in loader:
        xb = xb.to(device)
        optimizer.zero_grad()
        x_hat = model(xb)
        loss = criterion(x_hat, xb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, n = 0.0, 0
    for xb, _ in loader:
        xb = xb.to(device)
        loss = criterion(model(xb), xb)
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return total_loss / n

torch.manual_seed(42)
ae = Autoencoder(latent_dim=16).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(ae.parameters(), lr=1e-3)

EPOCHS = 10
train_losses, val_losses = [], []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tr = train_one_epoch(ae, train_loader, optimizer, criterion)
    va = evaluate(ae, test_loader, criterion)
    train_losses.append(tr); val_losses.append(va)
    print(f'Epoch {epoch:2d}/{EPOCHS} | train_mse={tr:.5f} | val_mse={va:.5f}')
print(f'\nTotal training time: {time.time() - t0:.1f}s')

In [ ]:
# Plot training curves
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='train')
plt.plot(val_losses,   label='val')
plt.title('Reconstruction MSE'); plt.xlabel('Epoch')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🖼️ 9. Visualizing Reconstructions

In [ ]:
# Show original vs reconstructed images
ae.eval()
xb, _ = next(iter(test_loader))
xb = xb.to(device)

with torch.no_grad():
    x_hat = ae(xb)

n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(16, 4))
for i in range(n_show):
    axes[0, i].imshow(xb[i].cpu().view(28, 28), cmap='gray')
    axes[0, i].set_title('original', fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(x_hat[i].cpu().view(28, 28), cmap='gray')
    axes[1, i].set_title('recon', fontsize=8)
    axes[1, i].axis('off')
plt.suptitle('Top: original   |   Bottom: reconstruction')
plt.tight_layout(); plt.show()

---
## 🧬 10. Extracting the Latent Representation

Sometimes the reconstruction is not the goal — the **latent vector** is.

```python
with torch.no_grad():
    latent = model.encoder(x)
```

```text
Input  → [batch, 784]
Latent → [batch, 16]
```

Uses:
- Clustering
- Visualization (t-SNE / UMAP)
- Classification
- Similarity search
- Anomaly detection

In [ ]:
# Extract latent features and visualize with PCA (2D)
ae.eval()
latents, labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        z = ae.encoder(xb.to(device)).cpu()
        latents.append(z)
        labels.append(yb)

Z = torch.cat(latents).numpy()
Y = torch.cat(labels).numpy()
print('Latent matrix:', Z.shape)

# Quick 2D projection via PCA (using numpy SVD)
Zc = Z - Z.mean(axis=0)
U, S, Vt = np.linalg.svd(Zc, full_matrices=False)
Z2 = Zc @ Vt[:2].T

plt.figure(figsize=(7, 6))
scatter = plt.scatter(Z2[:, 0], Z2[:, 1], c=Y, cmap='tab10', s=4, alpha=0.6)
plt.colorbar(scatter, label='digit')
plt.title('Latent space (16-dim → PCA 2D)')
plt.tight_layout(); plt.show()

---
## 🔬 11. Effect of Latent Size

Try `latent_dim ∈ {4, 16, 64}` and compare:

- Validation MSE
- Visual quality
- Training time

In [ ]:
def train_ae(latent_dim, epochs=5):
    torch.manual_seed(42)
    model = Autoencoder(latent_dim=latent_dim).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.MSELoss()
    t0 = time.time()
    for _ in range(epochs):
        train_one_epoch(model, train_loader, opt, crit)
    dt = time.time() - t0
    val = evaluate(model, test_loader, crit)
    return model, val, dt

results = {}
for ld in [4, 16, 64]:
    m, v, dt = train_ae(ld, epochs=5)
    results[ld] = (m, v, dt)
    print(f'latent_dim={ld:>3} | val_mse={v:.5f} | time={dt:.1f}s')

In [ ]:
# Compare reconstruction quality across latent sizes
xb, _ = next(iter(test_loader))
xb = xb.to(device)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i in range(8):
    axes[0, i].imshow(xb[i].cpu().view(28, 28), cmap='gray')
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_ylabel('original', fontsize=10)

for row, ld in enumerate([4, 16, 64], start=1):
    model = results[ld][0]
    model.eval()
    with torch.no_grad():
        rec = model(xb).cpu()
    for i in range(8):
        axes[row, i].imshow(rec[i].view(28, 28), cmap='gray')
        axes[row, i].axis('off')
    axes[row, 0].set_ylabel(f'latent={ld}', fontsize=10)

plt.suptitle('Reconstruction quality vs latent dimension')
plt.tight_layout(); plt.show()

---
## 🌫️ 12. Denoising Autoencoder

Instead of feeding clean input, we feed a **corrupted** version and ask the model to reconstruct the **clean** original:

```text
Clean image
    ↓
Add noise
    ↓
Noisy image
    ↓
Encoder → Decoder
    ↓
Clean reconstruction
```

This forces the network to learn **robust structure** rather than memorizing pixel values.

**Key trick:** the loss compares the **reconstruction against the clean input**, not the noisy one.

In [ ]:
# Train a denoising autoencoder
def train_denoising_epoch(model, loader, optimizer, criterion, noise_std=0.5):
    model.train()
    total_loss, n = 0.0, 0
    for xb, _ in loader:
        xb = xb.to(device)
        noisy = (xb + noise_std * torch.randn_like(xb)).clamp(0, 1)

        optimizer.zero_grad()
        x_hat = model(noisy)
        loss = criterion(x_hat, xb)   # <-- target is CLEAN input
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return total_loss / n

torch.manual_seed(42)
dae = Autoencoder(latent_dim=32).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(dae.parameters(), lr=1e-3)

for epoch in range(1, 8):
    loss = train_denoising_epoch(dae, train_loader, optimizer, criterion, noise_std=0.5)
    print(f'Epoch {epoch:2d}/7 | train_mse(clean target)={loss:.5f}')

In [ ]:
# Visualize: clean | noisy | denoised
dae.eval()
xb, _ = next(iter(test_loader))
xb = xb.to(device)
noisy = (xb + 0.5 * torch.randn_like(xb)).clamp(0, 1)

with torch.no_grad():
    denoised = dae(noisy)

n_show = 8
fig, axes = plt.subplots(3, n_show, figsize=(16, 6))
for i in range(n_show):
    axes[0, i].imshow(xb[i].cpu().view(28, 28), cmap='gray')
    axes[1, i].imshow(noisy[i].cpu().view(28, 28), cmap='gray')
    axes[2, i].imshow(denoised[i].cpu().view(28, 28), cmap='gray')
    for r in range(3):
        axes[r, i].axis('off')
axes[0, 0].set_ylabel('clean', fontsize=10)
axes[1, 0].set_ylabel('noisy', fontsize=10)
axes[2, 0].set_ylabel('denoised', fontsize=10)
plt.suptitle('Denoising autoencoder results')
plt.tight_layout(); plt.show()

---
## 🚨 13. Anomaly Detection

Train on **normal** data only. Then, at inference:

```text
Normal example
  ↓
Low reconstruction error

Anomalous example
  ↓
High reconstruction error
```

Threshold selection:

$$\text{error} > \text{threshold} \;\Rightarrow\; \text{anomaly}$$

⚠️ **Choose the threshold on validation data** — never arbitrarily, never on test data.

In [ ]:
# Simulate anomaly detection: train only on digit 1, then score ALL digits
normal_ds = Subset(train_ds, [i for i, (_, y) in enumerate(train_ds) if y == 1])
normal_loader = DataLoader(normal_ds, batch_size=128, shuffle=True, num_workers=2)
print('Normal training samples (digit 1):', len(normal_ds))

torch.manual_seed(42)
anomaly_ae = Autoencoder(latent_dim=8).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(anomaly_ae.parameters(), lr=1e-3)

for epoch in range(1, 8):
    loss = train_one_epoch(anomaly_ae, normal_loader, optimizer, criterion)
    print(f'Epoch {epoch:2d}/7 | train_mse(normal only)={loss:.5f}')

In [ ]:
# Score reconstruction errors for every digit
anomaly_ae.eval()
errors_per_digit = {d: [] for d in range(10)}

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        err = ((anomaly_ae(xb) - xb) ** 2).mean(dim=1)   # per-sample MSE
        for d in range(10):
            mask = (yb == d)
            if mask.any():
                errors_per_digit[d].extend(err[mask].cpu().tolist())

# Plot distribution of errors per digit
fig, ax = plt.subplots(figsize=(10, 4))
positions = list(errors_per_digit.keys())
data = [errors_per_digit[d] for d in positions]
ax.boxplot(data, labels=[str(d) for d in positions])
ax.set_xlabel('digit'); ax.set_ylabel('reconstruction MSE')
ax.set_title('Reconstruction error per digit (trained ONLY on 1)')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 📊 14. Autoencoder vs PCA

| Feature | PCA | Autoencoder |
| --- | --- | --- |
| Main method | Linear | Neural network |
| Nonlinear features | ❌ No | ✅ Yes |
| Complexity | Low | Higher |
| Interpretability | ✅ Higher | ❌ Lower |
| Large data | Limited | ✅ Strong |
| Deep representations | ❌ No | ✅ Yes |

A **linear** autoencoder (no nonlinear activation) learns a representation closely related to PCA. Nonlinear autoencoders capture more complex structure.

In [ ]:
# Compare PCA reconstruction MSE at various latent sizes (no sklearn needed)
xb, _ = next(iter(test_loader))
X = xb.numpy()   # [B, 784]

def pca_reconstruct_mse(X, k):
    mu = X.mean(axis=0, keepdims=True)
    Xc = X - mu
    # SVD-based PCA
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    components = Vt[:k]                       # [k, D]
    Z = Xc @ components.T                     # [B, k]
    X_rec = Z @ components + mu               # [B, D]
    return ((X - X_rec) ** 2).mean()

print(f'{"latent":>7} {"PCA MSE":>12}')
print('-' * 22)
for k in [4, 16, 64]:
    mse = pca_reconstruct_mse(X, k)
    print(f'{k:>7} {mse:>12.5f}')

print('\nCompare with the autoencoder validation MSE from the latent-size sweep above.')

---
## 🖼️ 15. Convolutional Autoencoder

Fully-connected autoencoders **flatten** images and lose spatial structure.

A **convolutional autoencoder** preserves space:

```text
Image
 ↓
Conv layers        ← encoder
 ↓
Latent representation
 ↓
Transpose Conv / Upsample   ← decoder
 ↓
Reconstructed image
```

Benefits:
- Preserves spatial structure
- Fewer parameters for image data
- Better reconstructions

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        # Encoder: 1x28x28 → 32x7x7 → latent
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),  nn.ReLU(),   # 28→14
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),   # 14→7
        )
        self.enc_fc = nn.Linear(32 * 7 * 7, latent_dim)
        self.dec_fc = nn.Linear(latent_dim, 32 * 7 * 7)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),   # 7→14
            nn.ConvTranspose2d(16, 1,  3, stride=2, padding=1, output_padding=1), nn.Sigmoid(), # 14→28
        )

    def forward(self, x):
        # x: [B, 1, 28, 28]
        h = self.encoder_conv(x)
        z = self.enc_fc(h.flatten(1))
        h = self.dec_fc(z).view(-1, 32, 7, 7)
        return self.decoder_conv(h)

# Shape check
cae = ConvAutoencoder(latent_dim=32)
x = torch.randn(4, 1, 28, 28)
print('Input :', x.shape)
print('Output:', cae(x).shape)
print('Params:', sum(p.numel() for p in cae.parameters()))

In [ ]:
# Load MNIST in image form for the conv autoencoder
conv_transform = transforms.ToTensor()   # keeps [1, 28, 28]
conv_train = datasets.MNIST(root='./data', train=True,  download=True, transform=conv_transform)
conv_test  = datasets.MNIST(root='./data', train=False, download=True, transform=conv_transform)

conv_train_loader = DataLoader(conv_train, batch_size=128, shuffle=True,  num_workers=2)
conv_test_loader  = DataLoader(conv_test,  batch_size=256, shuffle=False, num_workers=2)

def train_conv_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, n = 0.0, 0
    for xb, _ in loader:
        xb = xb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), xb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return total_loss / n

torch.manual_seed(42)
cae = ConvAutoencoder(latent_dim=32).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(cae.parameters(), lr=1e-3)

for epoch in range(1, 6):
    loss = train_conv_epoch(cae, conv_train_loader, optimizer, criterion)
    print(f'Epoch {epoch:2d}/5 | train_mse={loss:.5f}')

In [ ]:
# Visualize conv autoencoder reconstructions
cae.eval()
xb, _ = next(iter(conv_test_loader))
xb = xb.to(device)
with torch.no_grad():
    x_hat = cae(xb)

n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(16, 4))
for i in range(n_show):
    axes[0, i].imshow(xb[i].cpu().squeeze(),   cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(x_hat[i].cpu().squeeze(), cmap='gray'); axes[1, i].axis('off')
axes[0, 0].set_ylabel('original', fontsize=10)
axes[1, 0].set_ylabel('recon', fontsize=10)
plt.suptitle('Convolutional autoencoder reconstructions')
plt.tight_layout(); plt.show()

---
## ⚠️ 16. The Overcomplete Trap

If the latent dimension is **larger** than the input:

```text
Input  = 784
Latent = 2000
```

The model can learn a near-**identity mapping** — it just copies the input through.

This might reconstruct well but produces **no useful compressed representation**.

### Fixes

- Keep the latent **smaller** than the input
- Add **noise** (denoising autoencoder)
- Add **regularization** (dropout, weight decay)
- Use **sparse** autoencoders (penalize activations)

---
## 🎲 17. A Brief Connection to VAEs

A **Variational Autoencoder (VAE)** extends the basic autoencoder.

Instead of learning:

```text
x → fixed latent vector
```

it learns a **probabilistic latent distribution**:

```text
Input
 ↓
Encoder
 ↓
Distribution (μ, σ)
 ↓
Sample latent vector
 ↓
Decoder
 ↓
Reconstruction
```

VAEs are a major branch of **generative modeling** and deserve their own dedicated treatment. The key takeaway here:

> **A VAE is an autoencoder with a probabilistic latent space and a sampling step — perfect for generation, not just reconstruction.**

---
## 🚨 18. Common Mistakes

### ❌ 1. Wrong output activation
For normalized image data `[0, 1]`, use **sigmoid** at the output.  
For unnormalized/tabular data, use **no activation** (linear output).

### ❌ 2. Mismatched normalization
Input preprocessing and output activation must be **consistent**.

### ❌ 3. Latent dimension too large
Overcomplete autoencoders learn the identity and produce useless features.

### ❌ 4. Only checking training loss
Always inspect **validation/test** reconstruction — autoencoders can overfit too.

### ❌ 5. Equating good reconstruction with useful features
A model can reconstruct well without learning a representation that's useful for downstream tasks.

### ❌ 6. Thresholding anomalies on test data
Pick anomaly thresholds on **validation** data only.

---
## 🏋️ 19. Hands-On Exercises

### Exercise 1 — Latent Size Sweep
Train autoencoders with `latent_dim ∈ {2, 8, 32, 128}`. Plot val MSE vs latent size. Where does diminishing returns kick in?

### Exercise 2 — Deeper Encoder
Add another layer (e.g. `784 → 512 → 128 → 32 → latent`). Does it help or overfit?

### Exercise 3 — Sparsity Penalty
Add an L1 penalty on the latent activations:
```python
loss = mse + 1e-3 * z.abs().mean()
```
Visualize the resulting latent codes — do they become sparse?

### Exercise 4 — Denoising with Different Noise Levels
Try `noise_std ∈ {0.1, 0.3, 0.5, 0.8}`. Which gives the best clean reconstruction?

### Exercise 5 — Anomaly Detection Threshold
Choose a threshold from the validation set and report **precision / recall** on the test set for detecting non-1 digits.

### Exercise 6 — Convolutional Autoencoder for CIFAR-10
Extend the conv autoencoder to 3-channel inputs. Compare reconstruction quality for `latent_dim ∈ {32, 128}`.

### Exercise 7 — Latent Interpolation
Take two test images, encode them, and linearly interpolate between their latents. Decode intermediate points. What do you see?

---
## 🎤 20. Interview Questions

1. **What is an autoencoder?**  
   A network that learns to encode input into a representation and reconstruct the original input.

2. **What is the bottleneck?**  
   The compressed latent representation between encoder and decoder.

3. **What is reconstruction loss?**  
   A loss measuring the difference between original input and reconstruction (typically MSE).

4. **Why use an undercomplete representation?**  
   To force the model to learn a compact representation rather than trivially copying the input.

5. **What is a denoising autoencoder?**  
   A model that receives corrupted input and learns to reconstruct the clean version.

6. **How can autoencoders detect anomalies?**  
   Examples that reconstruct poorly have unusually high reconstruction error; a threshold flags them.

7. **Autoencoder vs PCA?**  
   PCA is linear; autoencoders can be nonlinear and learn deep representations.

8. **What is an overcomplete autoencoder?**  
   One whose latent dimension is larger than the input — prone to learning the identity.

9. **What does a VAE add?**  
   A probabilistic latent space and sampling — enabling generative modeling.

10. **Can you use the latent space for downstream tasks?**  
    Yes — it's a compact feature representation, useful for classification, clustering, and similarity search.

---
## ✅ 21. Knowledge Check

- [ ] Explain what an autoencoder is
- [ ] Describe encoder → latent → decoder
- [ ] Explain what a bottleneck enforces
- [ ] Compute MSE reconstruction loss by hand
- [ ] Build and train an autoencoder in PyTorch
- [ ] Extract the latent representation for downstream use
- [ ] Build a denoising autoencoder
- [ ] Use reconstruction error for anomaly detection
- [ ] Compare autoencoders with PCA
- [ ] Build a convolutional autoencoder
- [ ] Explain the overcomplete problem
- [ ] Explain at a high level what a VAE adds

---
## 🏁 22. Key Takeaways

### The autoencoder structure

```text
Autoencoder
│
├── Encoder
│      ↓
│   Latent Space
│      ↓
├── Decoder
│      ↓
└── Reconstruction
```

### Core objective

$$\boxed{x \;\rightarrow\; z \;\rightarrow\; \hat{x}}$$

with

$$\boxed{\hat{x} \approx x}$$

### Mental model

> **Encoder = compress / represent**  
> **Latent space = learned representation**  
> **Decoder = reconstruct**  
> **Reconstruction loss = how well the model rebuilt the input**

### The advanced family

```text
Autoencoder
    │
    ├── Undercomplete (bottleneck)
    ├── Denoising (noisy → clean)
    ├── Sparse (L1 on latents)
    ├── Convolutional (images)
    └── Variational (probabilistic) → generative models
```

### 🧭 Golden Rule

> **An autoencoder is only as useful as its latent space. Keep it undercomplete, regularize appropriately, and always validate reconstruction on unseen data.**

---

### 🔗 What's Next?

**Module 13 — GANs (Generative Adversarial Networks)**

Autoencoders reconstruct data. GANs **generate** new data by pitting a **generator** against a **discriminator** in an adversarial game — a different and powerful route to modern generative modeling.